In [1]:
import pathlib

import pandas as pd
from image_analysis_3D.file_utils.notebook_init_utils import init_notebook
from IPython.display import Markdown, display

root_dir, in_notebook = init_notebook()

In [ ]:
sc_profiles_path = pathlib.Path(
    root_dir,
    "data/all_patient_profiles/0.normalized_profiles/sc_norm_norm_profile.parquet",
).resolve(strict=True)
organoid_profiles_path = pathlib.Path(
    root_dir,
    "data/all_patient_profiles/0.normalized_profiles/organoid_norm_norm_profile.parquet",
).resolve(strict=True)

patient_extra_metadata_path = pathlib.Path(
    root_dir,
    "config/patient_extra_metadata/patient_drug_screen_theoretical_counts_and_tumor_type"
    ".tsv",
).resolve(strict=True)
table1_file_info_path = pathlib.Path(
    root_dir,
    "tables/results/table1_file_info.parquet",
).resolve(strict=True)
table2_results_path = pathlib.Path(
    root_dir,
    "tables/tables/table2.tsv",
).resolve()
table2_results_path.parent.mkdir(parents=True, exist_ok=True)

In [3]:
sc_df = pd.read_parquet(sc_profiles_path)
organoid_df = pd.read_parquet(organoid_profiles_path)

patient_extra_metadata_df = pd.read_csv(patient_extra_metadata_path, sep="\t")
file_info_df = pd.read_parquet(table1_file_info_path)

## Compound, treatment, well, well-FOV, organoid, and single-cell counts per patient

In [4]:
# get the unique combinations of Metadata_Biology_PatientTumor and Metadata_Experiment_Treatment
compounds_counts = (
    organoid_df.groupby(
        ["Metadata_Biology_PatientTumor", "Metadata_Experiment_Treatment"]
    )
    .size()
    .to_frame()
    .reset_index()
    .rename(columns={0: "count"})
    .drop(columns="count")
    .groupby(["Metadata_Biology_PatientTumor"])
    .size()
    .to_frame()
    .reset_index()
    .rename(columns={0: "number_of_compounds"})
)

In [5]:
# get the unique combinations of Metadata_Biology_PatientTumor, Metadata_Experiment_Treatment, and Metadata_Experiment_Dose
treatments_counts = (
    organoid_df.groupby(
        [
            "Metadata_Biology_PatientTumor",
            "Metadata_Experiment_Treatment",
            "Metadata_Experiment_Dose",
        ]
    )
    .size()
    .to_frame()
    .reset_index()
    .rename(columns={0: "count"})
    .drop(columns="count")
    .groupby(["Metadata_Biology_PatientTumor"])
    .size()
    .to_frame()
    .reset_index()
    .rename(columns={0: "number_of_treatments"})
)

In [6]:
# get the unique combinations of Metadata_Biology_PatientTumor and Metadata_Experiment_Well
well_counts = (
    sc_df.groupby(["Metadata_Biology_PatientTumor", "Metadata_Experiment_Well"])
    .size()
    .to_frame()
    .reset_index()
    .rename(columns={0: "count"})
    .drop(columns="count")
    .groupby(["Metadata_Biology_PatientTumor"])
    .size()
    .to_frame()
    .reset_index()
    .rename(columns={0: "number_of_wells"})
)

In [7]:
well_fov_counts = (
    sc_df.groupby(
        [
            "Metadata_Biology_PatientTumor",
            "Metadata_Experiment_Treatment",
            "Metadata_Experiment_Well",
            "Metadata_Experiment_WellFOV",
        ]
    )
    .size()
    .to_frame()
    .reset_index()
    .rename(columns={0: "count"})
    .drop(columns="count")
    .groupby(["Metadata_Biology_PatientTumor"])
    .size()
    .to_frame()
    .reset_index()
    .rename(columns={0: "number_of_well_fovs"})
)

In [ ]:
organoid_counts = (
    sc_df.groupby(
        [
            "Metadata_Biology_PatientTumor",
            "Metadata_Experiment_Treatment",
            "Metadata_Experiment_Well",
            "Metadata_Experiment_WellFOV",
            "Metadata_Object_ParentOrganoid",
        ]
    )
    .size()
    .to_frame()
    .reset_index()
    .rename(columns={0: "count"})
    .drop(columns="count")
)
organoid_counts = (
    organoid_counts.loc[organoid_counts["Metadata_Object_ParentOrganoid"] != -1]
    .groupby(["Metadata_Biology_PatientTumor"])
    .size()
    .to_frame()
    .reset_index()
    .rename(columns={0: "number_of_organoids"})
)

In [9]:
single_cell_counts = (
    sc_df.groupby(
        [
            "Metadata_Biology_PatientTumor",
            "Metadata_Experiment_Treatment",
            "Metadata_Experiment_Well",
            "Metadata_Experiment_WellFOV",
        ]
    )
    .size()
    .to_frame()
    .reset_index()
    .rename(columns={0: "number_of_single_cells"})
    .groupby(["Metadata_Biology_PatientTumor"])
    .sum()
    .drop(
        columns=[
            "Metadata_Experiment_Treatment",
            "Metadata_Experiment_Well",
            "Metadata_Experiment_WellFOV",
        ]
    )
    .reset_index()
)

In [10]:
table2 = pd.merge(
    pd.merge(
        pd.merge(
            pd.merge(
                pd.merge(
                    compounds_counts,
                    treatments_counts,
                    on="Metadata_Biology_PatientTumor",
                ),
                well_counts,
                on="Metadata_Biology_PatientTumor",
            ),
            well_fov_counts,
            on="Metadata_Biology_PatientTumor",
        ),
        organoid_counts,
        on="Metadata_Biology_PatientTumor",
    ),
    single_cell_counts,
    on="Metadata_Biology_PatientTumor",
)

table2 = pd.merge(
    table2,
    patient_extra_metadata_df,
    left_on="Metadata_Biology_PatientTumor",
    right_on="patient",
    how="left",
).drop(columns=["patient"])

# remove the NF0037CQ1 patient from the table
# this is a test patient and we don't want to include it in the analysis
# different microscope was used
table2 = table2.loc[
    table2["Metadata_Biology_PatientTumor"] != "NF0037_T1_CQ1"
].reset_index(drop=True)

## Aggregate the raw image file info (from table2) to the patient level

In [11]:
file_info_counts = (
    file_info_df.groupby("Patient")
    .agg(
        TotalImages=("z_dimension_size", "sum"),
        total_size_bytes=("file_size_bytes", "sum"),
    )
    .reset_index()
)
file_info_counts["TotalSize(TB)"] = (
    file_info_counts["total_size_bytes"] / (1024**4)
).round(2)
file_info_counts = file_info_counts.drop(columns=["total_size_bytes"])

## Combine patient/tumor level counts with image file counts and sizes

In [12]:
table2 = pd.merge(
    table2,
    file_info_counts,
    left_on="Metadata_Biology_PatientTumor",
    right_on="Patient",
    how="left",
).drop(columns=["Patient"])

In [13]:
tumor_type = table2.pop("Tumor_type")
table2.insert(1, "Tumor_type", tumor_type)
table2.rename(
    columns={
        "Metadata_Biology_PatientTumor": "Patient Tumor ",
        "Tumor_type": "Tumor type ",
        "number_of_compounds": "Compound Count",
        "number_of_treatments": "Treatment Count",
        "number_of_wells": "Well Count",
        "number_of_well_fovs": "Well FOV Count",
        "number_of_organoids": "Organoid Count",
        "number_of_single_cells": "Single Cell Count",
        "theoretical_number_of_compounds": "Theoretical Compound Count",
        "theoretical_number_of_treatments": "Theoretical Treatment Count",
        "theoretical_number_of_well_fovs": "Theoretical Well FOV Count",
        "TotalImages": "Total Image Count",
        "TotalSize(TB)": "Total Size (TB)",
    },
    inplace=True,
)

In [ ]:
table2 = table2.drop(
    columns=[
        "Compound Count",
        "Well Count",
        "Theoretical Compound Count",
        "Theoretical Treatment Count",
        "Theoretical Well FOV Count",
    ]
)

# add a total row to the table
total_row = pd.DataFrame(
    {
        "Patient Tumor ": ["Total"],
        "Tumor type ": ["-"],
        "Treatment Count": [table2["Treatment Count"].sum()],
        "Well FOV Count": [table2["Well FOV Count"].sum()],
        "Organoid Count": [table2["Organoid Count"].sum()],
        "Single Cell Count": [table2["Single Cell Count"].sum()],
        "Total Image Count": [table2["Total Image Count"].sum()],
        "Total Size (TB)": [table2["Total Size (TB)"].sum().round(2)],
    }
)
table2 = pd.concat([table2, total_row], ignore_index=True)
table2.to_csv(table2_results_path, index=False, sep="\t")
table2

In [15]:
# convert the table to a markdown table
table2_md = table2.to_markdown(index=False, tablefmt="pipe")

In [16]:
# Display as formatted markdown
print("Rendered Table:")
display(Markdown(table2_md))

Rendered Table:


| Patient Tumor    | Tumor type                                       |   Treatment Count |   Well FOV Count |   Organoid Count |   Single Cell Count |   Total Image Count |   Total size (TB) |
|:-----------------|:-------------------------------------------------|------------------:|-----------------:|-----------------:|--------------------:|--------------------:|------------------:|
| NF0014_T1        | Neurofibroma (subcutaneous)                      |                21 |               95 |              191 |                2132 |               17069 |              0.13 |
| NF0014_T2        | Plexiform Neurofibroma                           |                21 |              347 |              870 |                2939 |               42560 |              0.32 |
| NF0016_T1        | Neurofibroma, diffuse and plexiform              |                20 |              117 |              257 |                 981 |               13540 |              0.11 |
| NF0018_T6        | Cutaneous Neurofibroma                           |                21 |              152 |              271 |                1442 |               19260 |              0.14 |
| NF0021_T1        | Cutaneous Neurofibroma                           |                21 |              346 |              820 |                5027 |               42270 |              0.32 |
| NF0030_T1        | Myopericytoma                                    |                21 |              204 |              696 |                3755 |               19935 |              0.16 |
| NF0035_T1        | Cutaneous Neurofibroma                           |                21 |              347 |              621 |                4780 |               51985 |              0.4  |
| NF0037_T1        | Cutaneous Neurofibroma                           |                26 |              416 |              793 |                7489 |              185640 |              1.39 |
| NF0040_T1        | Schwannoma, with degeneration                    |                26 |              411 |              588 |                5006 |               58880 |              0.45 |
| NF0055_T1        | Plexiform Neurofibroma                           |                25 |              341 |              731 |                4325 |              231700 |              1.76 |
| SARCO219_T2      | MPNST in association with plexiform neurofibroma |                21 |              199 |             3086 |               12591 |               18575 |              0.14 |
| SARCO361_T1      | MPNST                                            |                21 |              349 |             1505 |                4029 |               39429 |              0.31 |
| Total            | -                                                |               265 |             3324 |            10429 |               54496 |              740843 |              5.63 |